# Logistic Regression - Interactive Visualizations

This notebook generates four key visualizations to build intuition about how logistic regression works:

1. **Decision Boundary** - How the model separates classes in 2D
2. **Sigmoid Function** - Mapping linear scores to probabilities
3. **Loss Landscape** - The convex optimization surface
4. **Convergence Curve** - Training progress over iterations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sys
sys.path.insert(0, '/workspace/ml-algorithms-deep-dive/logistic-regression/src')
from vectorized_implementation import LogisticRegressionVectorized, sigmoid

plt.style.use('seaborn-v0_8-whitegrid')

## Generate Synthetic Data

In [ ]:
np.random.seed(42)
m = 100

# Class 0: centered at (-1, -1)
X0 = np.random.randn(m//2, 2) * 0.8 + np.array([-1, -1])
y0 = np.zeros(m//2)

# Class 1: centered at (1, 1)
X1 = np.random.randn(m//2, 2) * 0.8 + np.array([1, 1])
y1 = np.ones(m//2)

X_vis = np.vstack([X0, X1])
y_vis = np.hstack([y0, y1])

print(f"Generated {len(X_vis)} samples with 2 features")
print(f"Class 0: {np.sum(y0==0)} samples")
print(f"Class 1: {np.sum(y1==1)} samples")

## Train Model

In [ ]:
model = LogisticRegressionVectorized(learning_rate=0.5, n_iterations=1000)
model.fit(X_vis, y_vis)

print(f"\nTrained weights: [{model.coef_[0]:.4f}, {model.coef_[1]:.4f}]")
print(f"Trained bias: {model.intercept_:.4f}")
print(f"Training accuracy: {model.score(X_vis, y_vis):.4f}")

## Visualization 1: Decision Boundary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Scatter plot by class
ax.scatter(X_vis[y_vis==0, 0], X_vis[y_vis==0, 1], 
           c='steelblue', s=60, alpha=0.7, label='Class 0', 
           edgecolors='white', linewidth=1.5)
ax.scatter(X_vis[y_vis==1, 0], X_vis[y_vis==1, 1], 
           c='coral', s=60, alpha=0.7, label='Class 1',
           edgecolors='white', linewidth=1.5)

# Create mesh for decision boundary
xx, yy = np.meshgrid(np.linspace(X_vis[:,0].min()-1, X_vis[:,0].max()+1, 200),
                     np.linspace(X_vis[:,1].min()-1, X_vis[:,1].max()+1, 200))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Plot decision boundary
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='-')

# Show probability contours
Z_prob = model.predict_proba(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
contour = ax.contourf(xx, yy, Z_prob, levels=10, cmap='RdBu_r', alpha=0.3)

ax.set_xlabel('Feature 1 (x₁)', fontsize=12)
ax.set_ylabel('Feature 2 (x₂)', fontsize=12)
ax.set_title('Decision Boundary: P(y=1|x) = 0.5', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')

# Colorbar for probability
cbar = plt.colorbar(contour, ax=ax)
cbar.set_label('P(Class 1)', fontsize=10)

plt.tight_layout()
plt.savefig('/workspace/ml-algorithms-deep-dive/logistic-regression/visualizations/01_decision_boundary.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 2: Sigmoid Function

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

z_range = np.linspace(-6, 6, 200)
sigmoid_vals = sigmoid(z_range)

ax.plot(z_range, sigmoid_vals, 'b-', linewidth=2.5, label='σ(z) = 1/(1+e⁻ᶻ)')
ax.axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Decision threshold')
ax.axvline(0, color='gray', linestyle=':', linewidth=1)
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axhline(1, color='gray', linestyle=':', linewidth=1)

# Mark key points
ax.scatter([0], [0.5], c='red', s=100, zorder=5)
ax.annotate('Threshold\n(50%)', xy=(0, 0.5), xytext=(1, 0.6),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10)

# Show sample predictions
sample_z = model.theta[0] + model.theta[1] * X_vis[:, 0] + model.theta[2] * X_vis[:, 1]
sample_probs = sigmoid(sample_z)

# Highlight misclassified points
misclassified = model.predict(X_vis) != y_vis
ax.scatter(sample_z[misclassified], sample_probs[misclassified], 
           c='red', s=80, marker='x', label='Misclassified', zorder=5)

# Highlight correct classifications
correct = model.predict(X_vis) == y_vis
ax.scatter(sample_z[correct], sample_probs[correct], 
           c='green', s=40, alpha=0.5, label='Correct', zorder=4)

ax.set_xlabel('Linear Score (z = wx + b)', fontsize=12)
ax.set_ylabel('Probability P(y=1|x)', fontsize=12)
ax.set_title('Sigmoid: Mapping Scores to Probabilities', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/workspace/ml-algorithms-deep-dive/logistic-regression/visualizations/02_sigmoid_function.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 3: Loss Landscape (3D)

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Grid for w1, w2 (fix bias at optimal)
w1_range = np.linspace(model.coef_[0] - 2, model.coef_[0] + 2, 40)
w2_range = np.linspace(model.coef_[1] - 2, model.coef_[1] + 2, 40)
W1, W2 = np.meshgrid(w1_range, w2_range)

Loss = np.zeros_like(W1)
for i in range(len(w1_range)):
    for j in range(len(w2_range)):
        test_theta = np.array([model.intercept_, W1[j,i], W2[j,i]])
        X_aug = np.column_stack([np.ones(len(X_vis)), X_vis])
        z_test = X_aug @ test_theta
        y_hat_test = sigmoid(z_test)
        epsilon = 1e-15
        y_hat_clipped = np.clip(y_hat_test, epsilon, 1 - epsilon)
        Loss[j,i] = -np.mean(y_vis * np.log(y_hat_clipped) + 
                            (1 - y_vis) * np.log(1 - y_hat_clipped))

surf = ax.plot_surface(W1, W2, Loss, cmap='viridis', alpha=0.8, edgecolor='none')

# Mark optimum
ax.scatter(model.coef_[0], model.coef_[1], 
           model.loss_history[-1], c='red', s=100, marker='*', 
           label=f'Optimum')

ax.set_xlabel('Weight w₁', fontsize=11)
ax.set_ylabel('Weight w₂', fontsize=11)
ax.set_zlabel('Cross-Entropy Loss', fontsize=11)
ax.set_title('Convex Loss Surface (No Local Minima)', fontsize=13, fontweight='bold')
ax.view_init(elev=25, azim=-60)
ax.legend()

plt.tight_layout()
plt.savefig('/workspace/ml-algorithms-deep-dive/logistic-regression/visualizations/03_loss_landscape.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualization 4: Convergence Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(model.loss_history, 'b-', linewidth=2)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title('Gradient Descent Convergence', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Annotate convergence
convergence_idx = np.argmin(np.abs(np.diff(model.loss_history)) < 0.0001) + 1
ax.axvline(convergence_idx, color='orange', linestyle='--', 
           label=f'Converged ~iter {convergence_idx}')

# Add inset showing final iterations
ax_inset = ax.inset_axes([0.55, 0.55, 0.4, 0.4])
ax_inset.plot(model.loss_history[-50:], 'b-', linewidth=1.5)
ax_inset.set_xlabel('Iteration', fontsize=8)
ax_inset.set_ylabel('Loss', fontsize=8)
ax_inset.set_title('Final 50 iters', fontsize=9)
ax_inset.grid(True, alpha=0.3)

ax.legend()

plt.tight_layout()
plt.savefig('/workspace/ml-algorithms-deep-dive/logistic-regression/visualizations/04_convergence_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary Statistics

In [ ]:
print("="*60)
print("LOGISTIC REGRESSION VISUALIZATION SUMMARY")
print("="*60)
print(f"\nModel Parameters:")
print(f"  Weights: [{model.coef_[0]:.4f}, {model.coef_[1]:.4f}]")
print(f"  Bias: {model.intercept_:.4f}")
print(f"\nPerformance:")
print(f"  Training Accuracy: {model.score(X_vis, y_vis):.4f}")
print(f"  Final Loss: {model.loss_history[-1]:.6f}")
print(f"  Initial Loss: {model.loss_history[0]:.6f}")
print(f"  Loss Reduction: {(model.loss_history[0] - model.loss_history[-1])/model.loss_history[0]*100:.1f}%")
print(f"\nVisualizations saved to:")
print(f"  - visualizations/01_decision_boundary.png")
print(f"  - visualizations/02_sigmoid_function.png")
print(f"  - visualizations/03_loss_landscape.png")
print(f"  - visualizations/04_convergence_curve.png")
print("\n" + "="*60)